# 10v5_00 — Test-like manifests (TL_EVAL, MOL_DEV, TL_1K ⊂ TL_3K ⊂ TL_10K, RND_1K)

From v5 on, **TL_EVAL selects and HOST confirms**. This notebook builds every query manifest
from observable, label-free test metadata. It never uses a HOST metric: HOST source and
structures are only **excluded**.

| manifest | purpose | rule |
|---|---|---|
| TL_EVAL | fixed model-selection population (~1,000) | timsTOF-type instrument, observed test adduct, test neutral-mass range, reconstructable mass, non-empty 50 ppm window; not HOST source/structure, not a DEV-1k structure, not a spectrum used to fit the frozen v4b mass model; 1 spectrum per connectivity; test-like weighted draw |
| MOL_DEV | molecule-aggregation development set (11_02) | test-like connectivities with ≥ 3 eligible spectra; disjoint from TL_EVAL |
| TL_1K ⊂ TL_3K ⊂ TL_10K | nested test-like training sets | prefixes of ONE test-like weighted permutation; disjoint from TL_EVAL, MOL_DEV, HOST |
| RND_1K | composition control | uniform draw from the same valid pool |

All draws are seed 42 and deterministic, which section 6 re-checks. This notebook also fixes
the v4b V1/V2 labeling retrospectively (section 7).

In [1]:
import sys, json, time
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'src').exists() and (REPO_ROOT.parent / 'src').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

import numpy as np
import pandas as pd

from casmi.paths import get_project_paths
from casmi.io.artifacts import load_artifact
from casmi.candidates.mass_index import MassIndex
from casmi.qcr.context import HOST_INGEST_LIB, SEED, v4b_paths, v5_paths
from casmi.ranking.manifests import (
    build_query_universe, candidate_counts, manifest_checks, manifest_fingerprint, save_manifest, select_mol_dev,
    select_tl_eval, select_training_manifests, test_like_frame, test_observable_profile, tl_eval_eligibility, training_pool,
)
from casmi.ranking.selection import effective_held_out_validity, filter_audit, no_op_duplicate_annotations
from casmi.ranking.training_sets import v2_filter, v3_filter
from casmi.validation.decision_log import log_decision

pd.set_option('display.max_columns', 60); pd.set_option('display.width', 200)
paths = get_project_paths()
vp, D = v4b_paths(paths), v5_paths(paths)
TL_EVAL_N, MOL_DEV_N_CONN = 1000, 400
TRAIN_SIZES = (('TL_1K', 1000), ('TL_3K', 3000), ('TL_10K', 10000))
with open(paths.processed / 'candidate_generation' / 'candidate_generation_contract.json') as f:
    SELECTED_PPM = json.load(f)['molecule_policy']['tolerance_ppm']
print('retrieval window (contract):', SELECTED_PPM, 'ppm')

retrieval window (contract): 50 ppm


## 1. Inputs: universe, observable test profile, exclusion sets

In [2]:
train_meta = load_artifact('train_spectrum_metadata')
folds = pd.read_parquet(paths.processed / 'connectivity_folds.parquet')
test_meta = pd.read_parquet(paths.processed / 'test_spectrum_metadata.parquet')
host_q = pd.read_parquet(vp.host_processed / 'host_holdout_spectra.parquet')
dev_1k = pd.read_parquet(paths.interim / 'spectral_ranking' / 'dev_ranking_queries.parquet')
dev_60k = pd.read_parquet(paths.interim / 'candidate_generation' / 'dev_queries.parquet', columns=['query_id', 'source'])
mmv = pd.read_parquet(paths.interim / 'candidate_generation' / 'molecule_mass_variants.parquet')
mass_index = MassIndex.from_dataframe(mmv, mass_col='exact_mass', key_col='mass_variant_id')

HOST_KEYS = set(host_q['true_connectivity_key'])
DEV1K_KEYS = set(dev_1k['true_connectivity_key'])
# every spectrum the frozen v4b mass-likelihood model was fitted on (60k dev_queries minus HOST minus DEV-1k)
MASS_FIT_IDS = set(dev_60k.loc[(dev_60k['source'] != HOST_INGEST_LIB) & ~dev_60k['query_id'].isin(set(dev_1k['query_id'])), 'query_id'])
profile = test_observable_profile(test_meta)
test_frame = test_like_frame(test_meta)
print(json.dumps({k: v for k, v in profile.items() if k != 'adducts_supported'}, indent=1))
print('observed test adducts:', len(profile['adducts']), '| unsupported by the parser:', [a for a, s in profile['adducts_supported'].items() if not s])
print(f'HOST structures: {len(HOST_KEYS)}  DEV-1k structures: {len(DEV1K_KEYS)}  v4b mass-fit spectra: {len(MASS_FIT_IDS):,}')

universe = build_query_universe(train_meta, folds)
universe['n_candidates_window'] = candidate_counts(universe, mass_index, SELECTED_PPM)
print(f'universe: {len(universe):,} train spectra with a connectivity key')

{
 "instruments": [
  "timsTOF"
 ],
 "adducts": [
  "[M+CH2O2-H]-",
  "[M+Cl]-",
  "[M+H]+",
  "[M+K]+",
  "[M+NH4]+",
  "[M+Na]+",
  "[M-H]-"
 ],
 "polarity_share": {
  "positive": 0.8136850783182193,
  "negative": 0.1863149216817807
 },
 "neutral_mass_min": 246.1003764519,
 "neutral_mass_max": 439.1037235481,
 "n_test_spectra": 1213,
 "n_test_molecules": 400
}
observed test adducts: 7 | unsupported by the parser: []
HOST structures: 250  DEV-1k structures: 986  v4b mass-fit spectra: 58,975
universe: 2,539,608 train spectra with a connectivity key


## 2. TL_EVAL: eligibility funnel and fixed selection

In [3]:
elig, criteria = tl_eval_eligibility(universe, profile, HOST_INGEST_LIB, HOST_KEYS, excluded_keys=DEV1K_KEYS, excluded_query_ids=MASS_FIT_IDS)
funnel, mask = [], np.ones(len(elig), bool)
for c in criteria:
    mask &= elig[f'crit_{c}'].to_numpy()
    funnel.append({'criterion (cumulative)': c, 'n_spectra': int(mask.sum()), 'n_connectivities': int(elig.loc[mask, 'connectivity_key'].nunique())})
display(pd.DataFrame(funnel))
tl_eval = select_tl_eval(elig, test_frame, n=TL_EVAL_N, seed=SEED)
print(f'TL_EVAL: {len(tl_eval)} queries (requested {TL_EVAL_N}); {tl_eval.connectivity_key.nunique()} connectivities')
meta_tl = save_manifest(tl_eval, 'TL_EVAL', D['manifests'], extra={'requested': TL_EVAL_N, 'profile': profile, 'seed': SEED,
                                                                   'criteria': criteria, 'funnel': funnel})
print('TL_EVAL sha256:', meta_tl['manifest_sha256'])

,criterion (cumulative),n_spectra,n_connectivities
0,timsTOF_like_instrument,1154969,183191
1,test_adduct,823111,179943
2,in_test_mass_range,817704,178770
3,neutral_mass_reconstructed,817704,178770
4,candidate_window_nonempty,817704,178770
5,not_host_source,817073,178631
6,not_host_structure,817073,178631
7,not_excluded_structure,814729,178187
8,not_excluded_query,795883,178139
9,has_fold,795883,178139


TL_EVAL: 1000 queries (requested 1000); 1000 connectivities
TL_EVAL sha256: dc77723e804c7548db9c85be63ae6ce40fe09398433b083bfe9d52b439c0eb98


## 3. MOL_DEV (molecule-level aggregation development set for 11_02)

In [4]:
elig_md, _ = tl_eval_eligibility(universe, profile, HOST_INGEST_LIB, HOST_KEYS,
                                 excluded_keys=DEV1K_KEYS | set(tl_eval['connectivity_key']), excluded_query_ids=MASS_FIT_IDS)
mol_dev = select_mol_dev(elig_md, test_frame, n_connectivities=MOL_DEV_N_CONN, min_spectra=3, seed=SEED)
print(f'MOL_DEV: {mol_dev.connectivity_key.nunique()} connectivities, {len(mol_dev)} spectra; spectra/connectivity:',
      mol_dev.groupby('connectivity_key').size().describe()[['min', '50%', 'max']].to_dict())
if mol_dev.connectivity_key.nunique() < 300:
    print('WARNING: fewer than 300 development molecules available -- recorded, not failed')
save_manifest(mol_dev, 'MOL_DEV', D['manifests'], extra={'requested_connectivities': MOL_DEV_N_CONN, 'min_spectra': 3, 'seed': SEED})

MOL_DEV: 400 connectivities, 1822 spectra; spectra/connectivity: {'min': 3.0, '50%': 4.0, 'max': 10.0}


{'name': 'MOL_DEV',
 'n_queries': 1822,
 'n_connectivities': 400,
 'manifest_sha256': 'dd22e377d3a1ec8114b0dc4ab7d46b99102dfe9fd9b2c8dc37fe94fb8b96334d',
 'written_at': '2026-09-24T22:26:27.539582+00:00',
 'requested_connectivities': 400,
 'min_spectra': 3,
 'seed': 42}

## 4. Training manifests (nested TL sets + RND_1K)

In [5]:
excluded_train_keys = set(tl_eval['connectivity_key']) | set(mol_dev['connectivity_key'])
pool = training_pool(universe, HOST_INGEST_LIB, HOST_KEYS, excluded_train_keys)
print(f'valid training pool: {len(pool):,} spectra, {pool.connectivity_key.nunique():,} connectivities')
train_manifests, counts, wreport = select_training_manifests(pool, test_frame, sizes=TRAIN_SIZES, rnd_size=1000, seed=SEED)
display(pd.DataFrame(counts).T)
for name, m in train_manifests.items():
    save_manifest(m, name, D['manifests'], extra={'requested': counts[name]['requested'], 'actual': counts[name]['actual'], 'seed': SEED})
wreport.to_csv(D['manifests'] / 'test_like_weight_report.csv', index=False)

# composition vs test (observable metadata only)
comp = []
for name, m in {'TL_EVAL': tl_eval, 'MOL_DEV': mol_dev, **train_manifests}.items():
    for dim in ('instrument', 'adduct', 'ionization_mode'):
        for lvl, share in m[dim].astype(str).value_counts(normalize=True).head(8).items():
            comp.append({'set': name, 'dim': dim, 'level': lvl, 'share': share})
comp = pd.DataFrame(comp)
comp.to_csv(D['manifests'] / 'composition_report.csv', index=False)
display(comp.pivot_table(index=['dim', 'level'], columns='set', values='share').round(3))

valid training pool: 2,459,951 spectra, 272,210 connectivities


,requested,actual
TL_1K,1000,1000
TL_3K,3000,3000
TL_10K,10000,10000
RND_1K,1000,1000


set                              MOL_DEV  RND_1K  TL_10K  TL_1K  TL_3K  TL_EVAL
dim             level                                                          
adduct          [2M+H]+              NaN   0.031     NaN    NaN    NaN      NaN
                [2M+Na]+             NaN   0.070     NaN    NaN    NaN      NaN
                [2M-H]-              NaN   0.026     NaN    NaN    NaN      NaN
                [M+CH2O2-H]-       0.021   0.031   0.013  0.010  0.012    0.020
                [M+Cl]-            0.001     NaN   0.000    NaN  0.000    0.002
                [M+H]+             0.752   0.518   0.874  0.883  0.883    0.843
                [M+K]+               NaN     NaN   0.000  0.001  0.000    0.004
                [M+NH4]+           0.010   0.025   0.001    NaN    NaN    0.002
                [M+Na]+            0.024   0.069   0.005  0.003  0.004    0.018
                [M-H]-             0.192   0.194   0.107  0.103  0.100    0.111
instrument      ESI-QFT              NaN   0.031   0.000    NaN    NaN      NaN
                LC-ESI-ITFT          NaN   0.017   0.001  0.001  0.001      NaN
                LC-ESI-QFT           NaN   0.022   0.002  0.002  0.001      NaN
                LC-ESI-QQ            NaN     NaN     NaN    NaN  0.001      NaN
                LC-ESI-QTOF          NaN   0.045   0.002  0.001  0.002      NaN
                Linear Ion Trap      NaN     NaN   0.000  0.001  0.001      NaN
                Orbitrap             NaN   0.292   0.013  0.013  0.013      NaN
                QTOF                 NaN   0.033   0.001  0.001  0.001      NaN
                qtof                 NaN   0.015     NaN    NaN    NaN      NaN
                timsTOF            1.000   0.467   0.980  0.981  0.979    1.000
ionization_mode negative           0.215   0.269   0.120  0.113  0.112    0.133
                positive           0.785   0.731   0.880  0.887  0.888    0.867

## 5. Structural checks (nesting, disjointness, HOST exclusion)

In [6]:
all_manifests = {'TL_EVAL': tl_eval, 'MOL_DEV': mol_dev, **train_manifests}
checks = manifest_checks(all_manifests, HOST_KEYS, HOST_INGEST_LIB)
display(checks)
assert checks['passed'].all(), checks[~checks['passed']]

,check,passed,detail
0,TL_1K ⊂ TL_3K,True,0 of TL_1K missing from TL_3K
1,TL_3K ⊂ TL_10K,True,0 of TL_3K missing from TL_10K
2,TL_1K ∩ TL_EVAL connectivity = ∅,True,0 shared
3,TL_1K ∩ MOL_DEV connectivity = ∅,True,0 shared
4,TL_1K ∩ HOST connectivity = ∅,True,0 shared
5,TL_1K excludes HOST source,True,
6,TL_3K ∩ TL_EVAL connectivity = ∅,True,0 shared
7,TL_3K ∩ MOL_DEV connectivity = ∅,True,0 shared
8,TL_3K ∩ HOST connectivity = ∅,True,0 shared
9,TL_3K excludes HOST source,True,


## 6. Deterministic regeneration under seed 42

In [7]:
tl_eval_2 = select_tl_eval(elig, test_frame, n=TL_EVAL_N, seed=SEED)
train_2, _, _ = select_training_manifests(pool, test_frame, sizes=TRAIN_SIZES, rnd_size=1000, seed=SEED)
regen = {'TL_EVAL': manifest_fingerprint(tl_eval_2) == manifest_fingerprint(tl_eval),
         **{n: manifest_fingerprint(train_2[n]) == manifest_fingerprint(train_manifests[n]) for n in train_manifests}}
print('regeneration identical:', regen)
assert all(regen.values())

regeneration identical: {'TL_EVAL': True, 'TL_1K': True, 'TL_3K': True, 'TL_10K': True, 'RND_1K': True}


## 7. v4b V1/V2 labeling fix (retrospective audit, no retraining)

A held-out-validity level counts only if its training restriction removed at least one query.
The historical model IDs are kept for reproducibility. A restriction that removed nothing is
annotated as a no-op duplicate, and its baseline is reported under the parent's name (V1).

In [8]:
dev_ids = set(dev_1k['query_id'])
v2_ids = set(v2_filter(dev_1k, HOST_INGEST_LIB)['query_id'])
v3_ids = set(v3_filter(dev_1k, HOST_INGEST_LIB, HOST_KEYS)['query_id'])
audits = {'source': filter_audit('source', dev_ids, v2_ids), 'structure': filter_audit('structure', v2_ids, v3_ids)}
eff = {m: effective_held_out_validity(m, audits) for m in ('V1', 'V2', 'V3', 'V4', 'V3_RD')}
dups = no_op_duplicate_annotations(audits)
record = {'training_filter_audits': audits, 'effective_held_out_validity': eff, 'no_op_duplicates': dups,
          'scientific_baseline_label': 'V1' if 'V2' in dups else 'V2 distinct from V1'}
display(pd.DataFrame(audits).T)
print(json.dumps({'effective_held_out_validity': eff, 'no_op_duplicates': dups}, indent=1))
(D['metrics'] / 'v4b_training_filter_audit.json').write_text(json.dumps(record, indent=2), encoding='utf-8')
if dups:
    log_decision(decision='v5: V2 annotated as a no-op duplicate of V1 (source filter removed 0 DEV training queries); baseline reported as V1',
                 evidence_used='outputs/v5/metrics/v4b_training_filter_audit.json', host_visible=False, category='bug_fix',
                 log_path=paths.outputs / 'decision_log.jsonl')

,filter,n_queries_before_filter,n_queries_after_filter,n_queries_removed,is_no_op
source,source,1000,1000,0,True
structure,structure,1000,973,27,False


{
 "effective_held_out_validity": {
  "V1": 0,
  "V2": 0,
  "V3": 1,
  "V4": 1,
  "V3_RD": 1
 },
 "no_op_duplicates": {
  "V2": "no-op duplicate of V1: every training restriction removed 0 queries"
 }
}


In [9]:
summary = {name: {'n_queries': int(len(m)), 'n_connectivities': int(m.connectivity_key.nunique()), 'sha256': manifest_fingerprint(m)}
           for name, m in all_manifests.items()}
(D['manifests'] / 'manifests_summary.json').write_text(json.dumps({'manifests': summary, 'profile': profile, 'seed': SEED,
                                                                   'checks_passed': bool(checks['passed'].all())}, indent=2, default=str), encoding='utf-8')
display(pd.DataFrame(summary).T)
print('next: python scripts/v5_build_scale_features.py --manifest TL_EVAL  (then TL_1K, RND_1K, TL_3K, TL_10K, MOL_DEV)')

,n_queries,n_connectivities,sha256
TL_EVAL,1000,1000,dc77723e804c7548db9c85be63ae6ce40fe09398433b08...
MOL_DEV,1822,400,dd22e377d3a1ec8114b0dc4ab7d46b99102dfe9fd9b2c8...
TL_1K,1000,996,1df919aea313c8c8c6bc8f4d7405d61f08273a6c617c5e...
TL_3K,3000,2973,efd0f2635b3288a9541f47890d7a73c76b75f0a976ae05...
TL_10K,10000,9722,7b7f42bd4af9550150e290a47574cd388e1fb4e12211cf...
RND_1K,1000,986,73e9135520db9d111dfa57af00ba3be878ff7adcc26210...


next: python scripts/v5_build_scale_features.py --manifest TL_EVAL  (then TL_1K, RND_1K, TL_3K, TL_10K, MOL_DEV)
